# Rider Counting — pipeline v2 (per-frame detection + near-frame dedup)

Notebook entry point for the v2 pipeline. The logic is imported directly from `scripts/run_rider_count_new.py` (single source, not a copy).

**Usage**: edit the configuration in Cell 2, then run the cells in order. Cells 3--4 run one location; use `batch_summary_new.ipynb` for the full batch.

In [ ]:
# Cell 1: Bootstrap — locate the repo root and load the pipeline module
import sys
import importlib.util
from pathlib import Path

p = Path.cwd().resolve()
while p != p.parent and not (p / "src").exists():
    p = p.parent
assert (p / "src").exists(), "Repo root with src/ not found — place this notebook inside the repo's notebooks/ folder"
REPO_ROOT = p
sys.path.insert(0, str(REPO_ROOT))
print("Repo root:", REPO_ROOT)

script = REPO_ROOT / "scripts" / "run_rider_count_new.py"
if not script.exists():
    hits = [h for h in REPO_ROOT.rglob("run_rider_count_new.py")]
    print("!! scripts/run_rider_count_new.py is not at the expected path")
    print("   found:", hits if hits else "(nowhere in the repo)")
    assert hits, "Extract the scripts/ folder to the repo root (next to src/)"
    script = hits[0]
    print("   using:", script)

spec = importlib.util.spec_from_file_location("run_rider_count_new", script)
rrc = importlib.util.module_from_spec(spec)
sys.modules["run_rider_count_new"] = rrc   # dataclasses need the module registered
spec.loader.exec_module(rrc)
print("Module loaded:", script.name)

In [ ]:
# Cell 2: Configuration — LOC is the only value you normally change
from types import SimpleNamespace

DATA_ROOT = Path(r"D:\0_MAIN_BIKE_DATASETS_clean")

LOC = "25"          # <- change only this: "01" ~ "27"; variants as "04-2" / "19-2"
MANUAL_COUNTS = None  # manual benchmark, e.g. (179, 62) = 179 with-flow / 62 against; keep None if absent

LOC_ID  = f"loc_{LOC}"
IMG_DIR = DATA_ROOT / f"Loc_{LOC}" / "Bicyclist"
ROI_JSON = REPO_ROOT / "configs" / "locations_new" / f"{LOC_ID}.json"
OUTDIR   = REPO_ROOT / "outputs_new" / LOC_ID

args = SimpleNamespace(
    model      = "yolov8s.pt",   # standard batch config; yolov8n.pt is faster but misses more
    imgsz      = 1280,           # high-resolution inference: key for night/small-target recall
    conf       = 0.10,
    classes    = {1},
    nms_iou    = 0.70,
    assoc_gap  = 3,
    min_move_px= 40.0,          # direction displacement gate: jitter <13px / real riders >50px; 40 sits in the empty band and blocks animal jitter
    cos_gate   = 0.5,
    # --- time / stationary / box-swap / recovery gates (mechanistic, independent of the manual counts) ---
    max_time_gap_s = 30.0,       # break association when EXIF capture times differ by >30s; 0 = off
    min_link_app_sim = 0.30,     # appearance gate on every association link (HSV fingerprint; color and brightness judged separately, min of the two)
    direction_max_span_s = 4.0,  # direction only from displacement within one burst (2s double-shot x2 margin)
    trust_rescue_pair_s = 0.0,   # experiment failed validation and was rolled back (queue artifact); keep 0. >0 is experimental
    second_pass = True,          # guided re-detection: missed 2s-pair frames re-detected at low conf + appearance check -> second_pass_dets.csv
    second_pass_conf = 0.05,     # second-pass confidence threshold
    stationary_filter = True,    # remove parked bicycles; removals stored in stationary_objects.csv for audit
    stationary_radius = 30.0,    # stationary cluster radius (px)
    stationary_hits   = 6,       # minimum distinct frames at the same spot
    stationary_span_s = 300.0,   # time-span threshold: nobody waits 5 minutes at a red light
    stationary_span_frames = 50, # frame-number span fallback when EXIF is absent
    stationary_min_density = 0.6,# a parked object appears in >=60% of its window's captures; busy chokepoints have low density and are kept
    reuse_detections = True,     # replay saved detections, skip YOLO; images are re-read to refresh HSV fingerprints
    save_crops = True,           # export rider crops (for orientation review) -> crops/
    save_viz   = True,           # export annotated visualizations -> viz/
    max_images = None,
)

print("LOC_ID:", LOC_ID)
print("IMG_DIR:", IMG_DIR, "| exists:", IMG_DIR.exists())
print("ROI_JSON:", ROI_JSON, "| exists:", ROI_JSON.exists())

In [ ]:
# Cell 3: Run the current location
summary = rrc.run_location(LOC_ID, IMG_DIR, ROI_JSON, OUTDIR, args)
summary

## Scene report

The next cells generate the full report for this location: the **QC funnel** (how many detections each stage removed and why), the **riders table**, the **person review queue** (sorted by confidence, for confirming missed riders by hand), and four publication-quality figures (also saved as 300 dpi PNGs to `outputs_new/loc_XX/report/`).

In [ ]:
# Cell 4: Report — tables
rep = rrc.generate_report(OUTDIR, ROI_JSON, LOC_ID, manual=MANUAL_COUNTS)

if "stats" in rep:
    display(rep["stats"])                  # scene overview: with/against counts, WW rate (CI), occupancy, vs manual count
display(rep["funnel"])                     # QC funnel: every step from raw captures to riders
if "riders_table" in rep:
    display(rep["riders_table"])           # per-rider detail
if "direction_table" in rep:
    display(rep["direction_table"])        # direction calls: each row matches one verification image in direction_check/
if "person_review" in rep:
    display(rep["person_review"])          # person candidates awaiting manual confirmation (crops in crops_person/)

In [ ]:
# Cell 5: Report figures (rendered inline; PNGs already saved to report/)
import matplotlib.pyplot as plt
plt.show()
print("Figure PNGs saved to:", OUTDIR / "report")

In [ ]:
# Cell 6: Generate the three-part manual review page (orientation labels + candidate confirmation + precision sample)
out, nr, nc = rrc.generate_review_html(OUTDIR, LOC_ID)
print(f"Review page written: {out}")
print(f"  Section A rider crops: {nr} (confirm authenticity + orientation)")
print(f"  Section B person candidates: {nc} (confirm missed riders)")
print("Open the html in a browser, label each item, click Export CSV; the batch notebook's metrics cell reads the exported review_*.csv")

In [ ]:
# Cell 7: Suspect static false-positive clusters (adjudication flow for issues like foliage detected as a bicycle)
# Lists recurring same-spot clusters that survived the automatic stationary filter -> verify them in crops/viz ->
# paste the confirmed zone_json rows into configs/locations_new/loc_XX.exclude.json -> re-run this location
sus = rrc.report_suspect_clusters(OUTDIR)
if len(sus):
    display(sus)
    print("How to adjudicate: each row is one suspect spot. Look up its rider_ids in crops/,")
    print("or inspect the (cx,cy) position in viz/ — once confirmed as foliage/static false positive,")
    print(f"write the zone_json content into {ROI_JSON.with_name(ROI_JSON.stem + '.exclude.json')}")
    print('Format: {"zones": [<zone_json1>, <zone_json2>, ...]} (replace note with the actual reason)')
    print("Then re-run Cell 3 (reuse_detections=True, tens of seconds) — excluded detections go to excluded_zone_dets.csv for audit")
else:
    print("No suspect static clusters — this location needs no exclusion zones")